# Treinamento e Avaliação de Modelos de Machine Learning

Este notebook realiza as seguintes etapas:
1. Carregar o dataset original e realizar o pré-processamento estruturado.
2. Treinar o `StandardScaler` e salvá-lo para uso futuro.
3. Dividir os dados em conjuntos de treino e teste.
4. Treinar e comparar três modelos de classificação:
   - Regressão Logística
   - Random Forest (Floresta Aleatória)
   - Gradient Boosting
5. Analisar a importância das features.
6. Salvar o melhor modelo treinado em disco para o simulador.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import joblib

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Carregamento e Preparação Completa dos Dados

In [ ]:
# Para garantir que o Scaler seja salvo junto com o modelo, faremos o pipeline de transformação aqui
df = pd.read_csv('../1_DataSet/Teen_Mental_Health_Dataset.csv')

# 1. Mapeamento de variáveis categóricas
df_ml = df.copy()
df_ml['genero'] = df_ml['genero'].map({'masculino': 0, 'feminino': 1})
df_ml['nivel_interacao_social'] = df_ml['nivel_interacao_social'].map({'baixo': 0, 'medio': 1, 'alto': 2})

# 2. One-hot encoding de uso_plataforma
df_ml = pd.get_dummies(df_ml, columns=['uso_plataforma'], prefix='plataforma')
plataforma_cols = [col for col in df_ml.columns if col.startswith('plataforma_')]
df_ml[plataforma_cols] = df_ml[plataforma_cols].astype(int)

# 3. Separando features (X) e target (y)
X = df_ml.drop(columns=['indicador_depressao'])
y = df_ml['indicador_depressao']

# 4. Ajustando e salvando o Scaler
cols_to_scale = [
    'idade', 'horas_diarias_redes_sociais', 'horas_sono', 
    'tempo_tela_antes_sono', 'desempenho_academico', 
    'atividade_fisica', 'nivel_estresse', 'nivel_ansiedade', 'nivel_vicio'
]

scaler = StandardScaler()
X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

# Salva o scaler para o simulador
joblib.dump(scaler, 'scaler.pkl')
print("Scaler salvo com sucesso como 'scaler.pkl'")

X.head()

## 2. Divisão em Treino e Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

## 3. Treinamento e Avaliação dos Modelos

In [ ]:
# Dicionário para armazenar as métricas
results = {}

def evaluate_model(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC': auc
    }
    
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    # Matriz de Confusão
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Matriz de Confusão - {name}')
    plt.xlabel('Predito')
    plt.ylabel('Real')
    plt.show()

### 3.1. Regressão Logística

In [ ]:
lr_model = LogisticRegression(random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)
evaluate_model(lr_model, 'Logistic Regression', X_test, y_test)

### 3.2. Random Forest

In [ ]:
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced', n_estimators=100)
rf_model.fit(X_train, y_train)
evaluate_model(rf_model, 'Random Forest', X_test, y_test)

### 3.3. Gradient Boosting

In [ ]:
gb_model = GradientBoostingClassifier(random_state=42, n_estimators=100)
gb_model.fit(X_train, y_train)
evaluate_model(gb_model, 'Gradient Boosting', X_test, y_test)

## 4. Comparação de Modelos

In [ ]:
df_results = pd.DataFrame(results).T
print(df_results)

# Gráfico comparativo de métricas
df_results.plot(kind='bar', figsize=(12, 6))
plt.title('Comparação de Métricas de Classificação')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

## 5. Análise de Importância das Features (Random Forest)

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(12, 6))
plt.title("Importância dos Atributos - Random Forest")
sns.barplot(x=importances[indices], y=features[indices], palette='viridis')
plt.xlabel('Importância Relativa')
plt.show()

## 6. Salvando o Melhor Modelo

In [ ]:
# Selecionamos o Random Forest como melhor modelo de escolha para o TCC,
# pois o Gradient Boosting atinge score perfeito (1.0) devido a regras determinísticas
# perfeitas do dataset sintético, o que pode configurar overfitting para dados reais.
best_model_name = 'Random Forest'
print(f"Modelo selecionado para exportação: {best_model_name}")

best_model = rf_model

# Salvando o modelo selecionado em disco
joblib.dump(best_model, 'best_model.pkl')
print(f"Modelo ({best_model_name}) salvo com sucesso como 'best_model.pkl'!")

## 7. Exportando Arquivos Auxiliares para o Power BI

In [ ]:
# 1. Exportar métricas comparativas dos modelos
df_results.to_csv('../5-ResultadosPBI/pbi_metricas_modelos.csv', index_label='Modelo')

# 2. Exportar importância de atributos
df_importances = pd.DataFrame({
    'Atributo': X.columns,
    'Importancia': importances
}).sort_values(by='Importancia', ascending=False)
df_importances.to_csv('../5-ResultadosPBI/pbi_importancia_atributos.csv', index=False)

# 3. Exportar dados de teste enriquecidos com indicadores híbridos e diagnósticos para o Power BI
df_test_export = X_test.copy()
df_test_export[cols_to_scale] = scaler.inverse_transform(df_test_export[cols_to_scale])

# Mapeamentos para exibição amigável no Power BI
df_test_export['genero'] = df_test_export['genero'].map({0: 'masculino', 1: 'feminino'})
df_test_export['nivel_interacao_social'] = df_test_export['nivel_interacao_social'].map({0: 'baixo', 1: 'medio', 2: 'alto'})
df_test_export['uso_plataforma'] = df.loc[X_test.index, 'uso_plataforma']

# Remover as colunas dadas por dummy, deixando apenas o uso_plataforma string
df_test_export = df_test_export.drop(columns=plataforma_cols)

# Adicionar realidades e predições
df_test_export['indicador_depressao_real'] = y_test
df_test_export['previsao_modelo_rf'] = rf_model.predict(X_test)
df_test_export['probabilidade_modelo_rf'] = rf_model.predict_proba(X_test)[:, 1]

# Funções internas para cálculo dos indicadores híbridos
def calc_qvl(row):
    sono = min(row['horas_sono'] / 9.0 * 10, 10)
    fisico = min(row['atividade_fisica'] / 2.0 * 10, 10)
    social = {'baixo': 2.0, 'medio': 6.0, 'alto': 10.0}.get(row['nivel_interacao_social'], 5.0)
    acad = min(row['desempenho_academico'] / 4.0 * 10, 10)
    return round((sono + fisico + social + acad) / 4.0, 2)

def calc_dd(row):
    redes = min(row['horas_diarias_redes_sociais'] / 8.0 * 10, 10)
    tela = min(row['tempo_tela_antes_sono'] / 3.0 * 10, 10)
    return round((redes + tela + row['nivel_vicio']) / 3.0, 2)

def calc_sp(row):
    return round((row['nivel_estresse'] + row['nivel_ansiedade']) / 2.0, 2)

def obter_rec(status):
    if status == 'Baixo Risco':
        return 'Paciente estável. A rotina apresenta bons fatores protetivos. Monitoramento padrão.'
    elif status == 'Risco Moderado':
        return 'Atenção Necessária. Tempo de tela elevado ou sono irregular. Higiene do sono.'
    else:
        return 'Alerta Crítico. Padrões de risco detectados com privação de sono e dependência digital. Recomendado apoio psicológico.'

df_test_export['qualidade_vida'] = df_test_export.apply(calc_qvl, axis=1)
df_test_export['dependencia_digital'] = df_test_export.apply(calc_dd, axis=1)
df_test_export['sobrecarga_psicologica'] = df_test_export.apply(calc_sp, axis=1)

# Calcular score de Risco Geral
df_test_export['risco_geral_percentual'] = (
    (df_test_export['probabilidade_modelo_rf'] * 100.0 * 0.40) + 
    (df_test_export['sobrecarga_psicologica'] * 10.0 * 0.40) + 
    ((10.0 - df_test_export['qualidade_vida']) * 10.0 * 0.10) + 
    (df_test_export['dependencia_digital'] * 10.0 * 0.10)
).round(1)

df_test_export['classificacao_risco'] = df_test_export['risco_geral_percentual'].apply(
    lambda x: 'Baixo Risco' if x < 30.0 else ('Risco Moderado' if x < 70.0 else 'Risco Crítico')
)
df_test_export['recomendacao_clinica'] = df_test_export['classificacao_risco'].apply(obter_rec)

df_test_export.to_csv('../5-ResultadosPBI/pbi_previsoes_teste.csv', index=False)

# 4. Exportar matriz de correlação em formato longo (unpivoted) para Heatmap no Power BI
df_corr_calc = df.copy()
df_corr_calc['genero'] = df_corr_calc['genero'].map({'masculino': 0, 'feminino': 1})
df_corr_calc['nivel_interacao_social'] = df_corr_calc['nivel_interacao_social'].map({'baixo': 0, 'medio': 1, 'alto': 2})
corr_matrix = df_corr_calc.corr(numeric_only=True)
corr_long = corr_matrix.stack().reset_index()
corr_long.columns = ['Atributo_A', 'Atributo_B', 'Correlacao']
corr_long.to_csv('../5-ResultadosPBI/pbi_matriz_correlacao.csv', index=False)

print("Arquivos de suporte ao Power BI enriquecidos e matriz de correlação gerados com sucesso na pasta 5-ResultadosPBI!")